# Voice-Cloning Evaluation — SIM & WER

Bewertet die generierten Audios von **Qwen3**, **Fish-Speech-S2** und **Voxtral** objektiv:

- **SIM** — Speaker-Similarity (WavLM-SV Cosine)
- **WER** — Word Error Rate (deutscher Whisper vs. Ground-Truth-Text)

Die Logik liegt im Paket `evaluation/`; dieses Notebook ruft es nur auf und visualisiert.

> Vorab: Audios gemäß `evaluation/README.md` ablegen und `evaluation/data/metadata.csv` pflegen.
> (Subjektive Metriken N-MOS/SMOS/Turing und Performance RTF/TTFA folgen später.)

In [ ]:
import sys
from pathlib import Path

# Projekt-Root (eine Ebene über notebooks/) auf den Pfad legen, damit `import evaluation` klappt
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from evaluation import config
from evaluation.dataset import coverage_report

print(f'Device: {config.DEVICE}')
print(f'Modelle: {config.MODELS}')

# Überblick: welche (id x Modell) liegen vor?
coverage_report()

## 1. Scoring

Berechnet SIM und WER je `(id x Modell)` und speichert `evaluation/results/results.csv`.

Beim ersten Lauf werden die Modelle heruntergeladen (Whisper-DE, WavLM-SV).

In [ ]:
from evaluation.run_eval import run, aggregate

df = run()
df.head()

In [ ]:
# Aggregat: Mittelwert | 95%-CI | n je Modell
aggregate(df)

## 2. Visualisierung

In [ ]:
import plotly.express as px

# Verteilung je Metrik und Modell (Box-Plots zeigen Spread, nicht nur Mittelwert)
for metric in [m for m in ['sim', 'wer'] if m in df.columns]:
    fig = px.box(df, x='model', y=metric, points='all', color='model',
                 title=f'{metric.upper()} je Modell')
    fig.show()

In [ ]:
# Trade-off Sprecherähnlichkeit vs. Verständlichkeit (oben links = ideal)
if {'sim', 'wer'}.issubset(df.columns):
    fig = px.scatter(df, x='wer', y='sim', color='model', hover_data=['id'],
                     title='SIM vs. WER (pro Utterance)')
    fig.show()